# Tâche 1 Inventaire des langues et ressources Wikimedia

**Projet 1 · African Language Data & LLM Readiness**
toutes les langues africaines disposant d'une édition Wikipédia, plus le français comme langue de comparaison.

**Consigne de la tâche** : inventorier les éditions linguistiques, le nombre d'articles, les dumps et APIs disponibles, les catégories et les projets Wikimedia pertinents. Documenter la date de collecte et la méthode de comptage. Mesurer la disponibilité réelle et signaler les limites plutôt que de forcer une couverture égale.

**Sources utilisées**
- Page officielle : https://www.wikipedia.org/ (portail des éditions)
- Bibliothèque `Wikipedia-API` : https://pypi.org/project/Wikipedia-API/ (lecture du contenu des pages)
- API MediaWiki (`api.php`) et API `sitematrix` de meta.wikimedia.org : la bibliothèque ne fournit ni la liste des éditions, ni les statistiques, ni les dumps
- Wikidata (SPARQL) : rattachement de chaque édition à sa langue
- dumps.wikimedia.org : disponibilité des dumps

**Étapes (chacune dans sa propre section)**
0. Configuration et journal de provenance
1. Liste de toutes les éditions Wikipédia
2. Sélection des éditions en langues africaines
3. Statistiques officielles de chaque édition, dont le nombre total de mots des articles
4. Autres projets Wikimedia par langue, Incubator compris
5. APIs disponibles
6. Dumps disponibles
7. Catégories principales
8. Répartition des longueurs d'articles (échantillon via Wikipedia-API)
9. Langues africaines en préparation dans l'Incubator (présentées à part)
10. Synthèse, contrôles automatiques et sauvegarde


## Étape 0 : Configuration et journal de provenance

- **`USER_AGENT`** : la [politique User-Agent de Wikimedia](https://meta.wikimedia.org/wiki/User-Agent_policy) impose un moyen de contact. donc j'ai utliser jerome.akota@afriklang.com
- **Provenance** : chaque requête est journalisée (URL, horodatage UTC, statut HTTP) et sa réponse brute est sauvegardée dans `data/raw/inventaire/<RUN_ID>/`.
- une requête à la fois, avec une pause, le paramètre et de nouvelles tentatives sur les erreurs 429 et 5xx.

In [1]:
%pip install Wikipedia-API

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import re
import sys
import time
import platform
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import quote

import pandas as pd
import requests
import yaml
import wikipediaapi


USER_AGENT = "projet_stage_nlp/0.1 (contact: jerome.akota@afriklang.com)" 
PAUSE_S = 0.2                 
SAMPLE_SIZE = 30               
SEUIL_COURT_MOTS = 100        
CATEGORIES_TOP_N = 15          
CATEGORIES_MAX_REQUESTS = 20   
INCUBATOR_MAX_REQUESTS = 5     

assert "A_RENSEIGNER" not in USER_AGENT, "Renseigne un contact (e-mail) dans USER_AGENT avant d'exécuter."

#Chemins 
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CONFIG = yaml.safe_load(open(ROOT / "config" / "config.yaml", encoding="utf-8"))
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RAW_DIR = ROOT / CONFIG["paths"]["raw"] / "inventaire" / RUN_ID
OUT_DIR = ROOT / CONFIG["paths"]["processed"] / "inventaire" / RUN_ID
RAW_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

REFERENCE = CONFIG["scope"]["reference_language"]      
INITIALES = CONFIG["scope"]["initial_languages"]       

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 80)
print("RUN_ID :", RUN_ID)
print("Réponses brutes :", RAW_DIR)
print("Résultats       :", OUT_DIR)

RUN_ID : 20261023T132917Z
Réponses brutes : C:\Users\akota\desktop\projet_stage_nlp\data\raw\inventaire\20261023T132917Z
Résultats       : C:\Users\akota\desktop\projet_stage_nlp\data\processed\inventaire\20261023T132917Z


In [3]:
# Client HTTP avec journal de provenance 
SESSION = requests.Session()
SESSION.headers.update({"User-Agent": USER_AGENT})
PROVENANCE = []  

def _now():
    return datetime.now(timezone.utc).isoformat(timespec="seconds")


def _save_raw(n, label, body, is_json):
    """Sauvegarde la réponse brute et renvoie le nom du fichier."""
    safe = re.sub(r"[^A-Za-z0-9_.-]+", "_", label)[:80]
    path = RAW_DIR / f"{n:05d}_{safe}.{'json' if is_json else 'txt'}"
    with open(path, "w", encoding="utf-8") as f:
        if is_json:
            json.dump(body, f, ensure_ascii=False)
        else:
            f.write(body)
    return path.name


def get(url, params=None, label="requete", expect_json=True, max_retries=5):
    """GET avec nouvelles tentatives, maxlag et journalisation. Renvoie None en cas d'échec."""
    params = dict(params or {})
    if url.endswith("api.php"):
        params.setdefault("format", "json")
        params.setdefault("formatversion", "2")
        params.setdefault("maxlag", "5")
    last_status, last_error = None, None
    for attempt in range(max_retries):
        ts = _now()
        try:
            r = SESSION.get(url, params=params, timeout=60)
        except requests.RequestException as e:
            last_error = str(e)
            time.sleep(2 ** attempt)
            continue
        last_status = r.status_code
        retry_after = r.headers.get("Retry-After", "")
        wait = int(retry_after) if retry_after.isdigit() else 2 ** attempt
        if r.status_code in (429, 500, 502, 503, 504):
            time.sleep(wait)
            continue
        if r.status_code != 200:
            break
        try:
            body = r.json() if expect_json else r.text
        except ValueError as e:
            last_error = f"JSON invalide : {e}"
            break
        if expect_json and isinstance(body, dict) and isinstance(body.get("error"), dict) \
                and body["error"].get("code") == "maxlag":
            time.sleep(wait)
            continue
        n = len(PROVENANCE)
        PROVENANCE.append({"n": n, "label": label, "url": r.url, "horodatage_utc": ts,
                           "statut_http": r.status_code,
                           "fichier_brut": _save_raw(n, label, body, expect_json)})
        time.sleep(PAUSE_S)
        return body
    PROVENANCE.append({"n": len(PROVENANCE), "label": label, "url": url, "params": params,
                       "horodatage_utc": _now(), "statut_http": last_status,
                       "erreur": last_error or f"HTTP {last_status}", "fichier_brut": None})
    return None


def api(base_url, label, **params):
    """Appel à l'API MediaWiki d'un wiki (base_url = https://xx.wikipedia.org)."""
    return get(f"{base_url}/w/api.php", params, label=label)


def sparql(query, label):
    """Requête SPARQL sur le Wikidata Query Service."""
    return get("https://query.wikidata.org/sparql", {"query": query, "format": "json"}, label=label)


def sparql_rows(data):
    """Convertit une réponse SPARQL JSON en liste de dictionnaires simples."""
    if not data:
        return []
    return [{k: v["value"] for k, v in b.items()} for b in data["results"]["bindings"]]


def qid(uri):
    return uri.rsplit("/", 1)[-1] if isinstance(uri, str) else uri

## Étape 1: Liste de toutes les éditions Wikipédia

Le portail https://www.wikipedia.org/ n'affiche que les éditions d'au moins 100 articles. La liste exhaustive, éditions fermées comprises, vient de l'API `sitematrix` de meta.wikimedia.org (`smstate=all`).
Le même appel donne aussi les autres projets de chaque langue (Wiktionnaire, Wikibooks, etc.), qui servent à l'étape 4.

In [4]:
sm = api("https://meta.wikimedia.org", "sitematrix",
         action="sitematrix", smtype="language", smstate="all",
         smsiteprop="url|dbname|code|sitename")
assert sm is not None, "Échec de l'appel sitematrix : voir PROVENANCE[-1]"

rows = []
for key, lang in sm["sitematrix"].items():
    if key == "count":
        continue
    for site in lang.get("site", []):
        rows.append({
            "code": lang["code"],
            "nom_anglais": lang.get("localname"),
            "nom_local": lang.get("name"),
            "projet": site["code"],           # wiki, wiktionary, wikibooks, ...
            "dbname": site["dbname"],
            "url": site["url"],
            "etat": "fermé" if site.get("closed") else "ouvert",
        })
projets = pd.DataFrame(rows)
wikipedias = projets[projets["projet"] == "wiki"].reset_index(drop=True)

print("Éditions Wikipédia :", len(wikipedias))
print(wikipedias["etat"].value_counts().to_string())
wikipedias.head()

Éditions Wikipédia : 364
etat
ouvert    348
fermé      16


,code,nom_anglais,nom_local,projet,dbname,url,etat
0,aa,Afar,Qafár af,wiki,aawiki,https://aa.wikipedia.org,fermé
1,ab,Abkhazian,аԥсшәа,wiki,abwiki,https://ab.wikipedia.org,ouvert
2,ace,Acehnese,Acèh,wiki,acewiki,https://ace.wikipedia.org,ouvert
3,ady,Adyghe,адыгабзэ,wiki,adywiki,https://ady.wikipedia.org,ouvert
4,af,Afrikaans,Afrikaans,wiki,afwiki,https://af.wikipedia.org,ouvert


## Étape 2: Sélection des éditions en langues africaines

**Critère automatique (Wikidata)** : une édition Wikipédia (Q10876391) est rattachée à une langue (P407). Elle est sélectionnée si cette langue est « indigène à » (P2341), ou associée par « pays » (P17), un lieu situé en Afrique (continent Q15).

**Revue manuelle**, documentée ligne par ligne dans le tableau :
- **exclusion** des langues d'origine non africaine, présentes en Afrique par la colonisation ou par une diaspora, et des faux positifs ;
- **ajout** des langues africaines que la requête ne détecte pas le peul, sans lieu « indigène à » dans Wikidata, et les éditions fermées dont les lieux sont des régions plutôt que des pays ;
- **référence** : le français, conservé comme langue de comparaison.

Si Wikidata détecte une **nouvelle** langue absente de ces listes, elle est incluse par défaut et signalée  à revoir .

In [5]:
# Langue de chaque édition (toutes éditions)
q_editions = """
SELECT ?code ?lang ?iso3 ?nom_fr WHERE {
  ?wp wdt:P31 wd:Q10876391 ; wdt:P424 ?code .
  OPTIONAL { ?wp wdt:P407 ?lang .
             OPTIONAL { ?lang wdt:P220 ?iso3 . }
             OPTIONAL { ?lang rdfs:label ?nom_fr . FILTER(LANG(?nom_fr) = "fr") } }
}"""
# Éditions dont la langue est rattachée à l'Afrique
q_afrique = """
SELECT ?code ?lang ?prop WHERE {
  ?wp wdt:P31 wd:Q10876391 ; wdt:P424 ?code ; wdt:P407 ?lang .
  { ?lang wdt:P2341 ?place . BIND("P2341" AS ?prop) } UNION { ?lang wdt:P17 ?place . BIND("P17" AS ?prop) }
  { ?place wdt:P30 wd:Q15 } UNION { ?place wdt:P17 ?pays . ?pays wdt:P30 wd:Q15 }
}"""

wd_editions = pd.DataFrame(sparql_rows(sparql(q_editions, "wikidata_editions_langues")))
wd_afrique = pd.DataFrame(sparql_rows(sparql(q_afrique, "wikidata_editions_afrique")))
assert len(wd_editions) and len(wd_afrique), "Échec Wikidata : voir PROVENANCE"

langue_par_code = (wd_editions.assign(lang=wd_editions["lang"].map(qid))
                   .groupby("code").agg(qid_langue=("lang", lambda s: "|".join(sorted(set(s.dropna())))),
                                        iso639_3=("iso3", lambda s: "|".join(sorted(set(s.dropna())))),
                                        nom_francais=("nom_fr", lambda s: " / ".join(sorted(set(s.dropna()))))))
detectes = wd_afrique.groupby("code")["prop"].agg(lambda s: "|".join(sorted(set(s))))
print("Éditions détectées par Wikidata :", len(detectes))

Éditions détectées par Wikidata : 71


In [6]:
# Décisions manuelles (revue du 23/10/2026) 
EXCLUSIONS = {
    "en":  "langue d'origine non africaine (présente en Afrique par la colonisation)",
    "es":  "langue d'origine non africaine (présente en Afrique par la colonisation)",
    "pt":  "langue d'origine non africaine (présente en Afrique par la colonisation)",
    "pa":  "langue d'origine non africaine (présente en Afrique par une diaspora)",
    "pnb": "langue d'origine non africaine (présente en Afrique par une diaspora)",
    "ta":  "langue d'origine non africaine (présente en Afrique par une diaspora)",
    "yi":  "langue d'origine non africaine (présente en Afrique par une diaspora)",
    "lad": "langue d'origine non africaine (présente en Afrique par une diaspora)",
    "sl":  "faux positif de la requête Wikidata (langue européenne)",
}
AJOUTS = {
    "ff": "ajout manuel : aucun lieu « indigène à » renseigné dans Wikidata",
    "aa": "ajout manuel : édition fermée, Corne de l'Afrique",
    "ak": "ajout manuel : édition fermée, Ghana",
    "hz": "ajout manuel : édition fermée, Namibie et Botswana",
    "kj": "ajout manuel : édition fermée, Namibie et Angola",
    "ng": "ajout manuel : édition fermée, Namibie",
    "kr": "ajout manuel : édition fermée, Afrique de l'Ouest et du Nord",
}
A_DECIDER = {
    "ar": "arabe standard moderne : origine péninsule arabique, langue officielle en Afrique du Nord — à trancher avec l'encadrant",
}
REVUES = set(EXCLUSIONS) | set(AJOUTS) | set(A_DECIDER) | {REFERENCE} | {
    "af", "am", "ann", "ary", "arz", "bm", "bol", "dag", "dga", "din", "ee", "fat", "fon", "gpe", "gur",
    "guw", "ha", "ig", "igl", "kab", "kai", "kaj", "kbp", "kcg", "kg", "ki", "knc", "kus", "lg", "ln",
    "mg", "mos", "nqo", "nr", "nso", "nup", "ny", "om", "pcm", "rn", "rw", "sg", "shi", "sn", "so", "ss",
    "st", "sw", "ti", "tig", "tn", "ts", "tum", "tw", "ve", "wo", "xh", "yo", "zgh", "zu",
}

def decider(code):
    if code == REFERENCE:
        return "référence", "langue de comparaison"
    if code in EXCLUSIONS:
        return "exclu", EXCLUSIONS[code]
    if code in A_DECIDER:
        return "à décider", A_DECIDER[code]
    if code in AJOUTS:
        return "inclus", AJOUTS[code]
    if code not in REVUES:
        return "inclus", "nouvelle détection Wikidata — à revoir"
    return "inclus", ""

candidats = sorted(set(detectes.index) | set(AJOUTS) | {REFERENCE})
selection = wikipedias[wikipedias["code"].isin(candidats)].copy()
selection = selection.merge(langue_par_code, left_on="code", right_index=True, how="left")
selection["critere"] = selection["code"].map(lambda c: f"wikidata:{detectes[c]}" if c in detectes.index else "manuel")
selection[["decision", "commentaire"]] = selection["code"].apply(lambda c: pd.Series(decider(c)))
selection["langue_initiale"] = selection["code"].isin(INITIALES)

manquants = sorted(set(candidats) - set(selection["code"]))
if manquants:
    print("Codes absents de la sitematrix :", manquants)

print(pd.crosstab(selection["decision"], selection["etat"], margins=True, margins_name="total").to_string())
a_revoir = selection[selection["commentaire"].str.contains("à revoir")]
if len(a_revoir):
    print("\n⚠ Nouvelles détections à revoir :", a_revoir["code"].tolist())

# Éditions mesurées dans la suite : incluses + à décider + référence
editions = selection[selection["decision"] != "exclu"].sort_values("code").reset_index(drop=True)
print("\nÉditions inventoriées :", len(editions))
selection.sort_values(["decision", "code"])[
    ["code", "nom_francais", "nom_local", "nom_anglais", "etat", "qid_langue", "iso639_3",
     "critere", "decision", "commentaire"]]

etat       fermé  ouvert  total
decision                       
exclu          0       9      9
inclus         6      61     67
référence      0       1      1
à décider      0       1      1
total          6      72     78

Éditions inventoriées : 69


,code,nom_francais,nom_local,nom_anglais,etat,qid_langue,iso639_3,critere,decision,commentaire
82,en,anglais,English,English,ouvert,Q1860,eng,wikidata:P17,exclu,langue d'origine non africaine (présente en Afrique par la colonisation)
84,es,espagnol,español,Spanish,ouvert,Q1321,spa,wikidata:P17,exclu,langue d'origine non africaine (présente en Afrique par la colonisation)
180,lad,judéo-espagnol,Ladino,Ladino,ouvert,Q36196,lad,wikidata:P17,exclu,langue d'origine non africaine (présente en Afrique par une diaspora)
248,pa,pendjabi,ਪੰਜਾਬੀ,Punjabi,ouvert,Q58635,pan,wikidata:P17,exclu,langue d'origine non africaine (présente en Afrique par une diaspora)
260,pnb,pendjabi,پنجابی,Western Punjabi,ouvert,Q58635,pan,wikidata:P17,exclu,langue d'origine non africaine (présente en Afrique par une diaspora)
264,pt,portugais,português,Portuguese,ouvert,Q5146,por,wikidata:P17,exclu,langue d'origine non africaine (présente en Afrique par la colonisation)
295,sl,slovène,slovenščina,Slovenian,ouvert,Q9063,slv,wikidata:P17,exclu,faux positif de la requête Wikidata (langue européenne)
312,ta,tamoul,தமிழ்,Tamil,ouvert,Q5885,tam,wikidata:P17,exclu,langue d'origine non africaine (présente en Afrique par une diaspora)
356,yi,yiddish,ייִדיש,Yiddish,ouvert,Q8641,yid,wikidata:P17,exclu,langue d'origine non africaine (présente en Afrique par une diaspora)
0,aa,NaN,Qafár af,Afar,fermé,NaN,NaN,manuel,inclus,"ajout manuel : édition fermée, Corne de l'Afrique"


## Étape 3: Statistiques officielles de chaque édition

Ce sont les chiffres affichés sur la page **Spécial Statistiques** de chaque édition (vérifié sur la Wikipédia en wolof le 23/10/2026 : valeurs identiques). On les lit par l'API, `action=query&meta=siteinfo&siprop=general|statistics|rightsinfo|usergroups&sinumberingroup=1`, car les libellés de la page changent d'une langue à l'autre. Ce sont des valeurs en direct, horodatées dans le journal de provenance.

**Définitions (MediaWiki)** :
- **articles** : pages de l'espace principal (namespace 0) qui ne sont pas des redirections et qui contiennent au moins un lien interne. Un article d'une phrase compte autant qu'un article de 5 000 mots.
- **mots des articles** (`cirrussearch-article-words`) : nombre **total** de mots de tous les articles, calculé par le moteur de recherche CirrusSearch à partir du texte des articles. C'est une mesure exhaustive du volume de texte, pas une estimation. La méthode de découpage en mots est celle de CirrusSearch.
- **pages** : toutes les pages de tous les espaces de noms (discussions, modèles, utilisateurs…).
- **modifications par page** : nombre total de modifications divisé par le nombre de pages.
- **utilisateurs actifs** : comptes ayant fait au moins une action dans les 30 derniers jours.
- **administrateurs** et **bots** : nombre de comptes dans les groupes `sysop` et `bot`. Un nombre élevé de bots peut signaler des articles créés automatiquement.

In [7]:
stats = []
for _, e in editions.iterrows():
    d = api(e["url"], f"siteinfo_{e['code']}", action="query", meta="siteinfo",
            siprop="general|statistics|rightsinfo|usergroups", sinumberingroup="1")
    if d is None:
        stats.append({"code": e["code"], "erreur_stats": "échec requête"})
        continue
    s, g, ri = d["query"]["statistics"], d["query"]["general"], d["query"].get("rightsinfo", {})
    groupes = {grp.get("name"): grp.get("number") for grp in d["query"].get("usergroups", [])}
    mots = s.get("cirrussearch-article-words")
    stats.append({
        "code": e["code"],
        "articles": s.get("articles"),
        "mots_articles": mots,
        "mots_par_article": round(mots / s["articles"], 1) if mots and s.get("articles") else None,
        "pages": s.get("pages"),
        "modifications_total": s.get("edits"),
        "modifications_par_page": round(s["edits"] / s["pages"], 2) if s.get("pages") else None,
        "utilisateurs": s.get("users"), "utilisateurs_actifs": s.get("activeusers"),
        "administrateurs": s.get("admins", groupes.get("sysop")), "bots": groupes.get("bot"),
        "fichiers": s.get("images"),
        "langue_contenu": g.get("lang"), "version_mediawiki": g.get("generator"),
        "licence": ri.get("text"), "licence_url": ri.get("url"),
        "date_stats_utc": PROVENANCE[-1]["horodatage_utc"],
    })
stats = pd.DataFrame(stats)
manque_mots = stats.loc[stats["mots_articles"].isna(), "code"].tolist() if "mots_articles" in stats else []
if manque_mots:
    print("⚠ Nombre de mots non fourni par l'API pour :", manque_mots)
stats.sort_values("mots_articles", ascending=False)

,code,articles,mots_articles,mots_par_article,pages,modifications_total,modifications_par_page,utilisateurs,utilisateurs_actifs,administrateurs,bots,fichiers,langue_contenu,version_mediawiki,licence,licence_url,date_stats_utc
17,fr,2786157,1932510322,693.6,14183897,240165619,16.93,6120174,37142,143,63,79318,fr,MediaWiki 1.47.0-wmf.24,Creative Commons Attribution-Share Alike 4.0,https://creativecommons.org/licenses/by-sa/4.0/deed.fr,2026-10-23T13:29:36+00:00
5,ar,1338075,506530597,378.6,9237733,76925195,8.33,3015619,5575,25,19,56528,ar,MediaWiki 1.47.0-wmf.24,Creative Commons Attribution-Share Alike 4.0,https://creativecommons.org/licenses/by-sa/4.0/deed.ar,2026-10-23T13:29:26+00:00
7,arz,1633890,269346473,164.8,2270142,13274257,5.85,297668,290,7,34,1302,arz,MediaWiki 1.47.0-wmf.24,Creative Commons Attribution-Share Alike 4.0,https://creativecommons.org/licenses/by-sa/4.0/deed.arz,2026-10-23T13:29:28+00:00
21,ha,121191,77392768,638.6,169237,970436,5.73,49842,286,18,15,0,ha,MediaWiki 1.47.0-wmf.24,Creative Commons Attribution-Share Alike 4.0,https://creativecommons.org/licenses/by-sa/4.0/deed.ha,2026-10-23T13:29:40+00:00
1,af,138988,47372484,340.8,470878,3053681,6.49,214083,273,15,65,10259,af,MediaWiki 1.47.0-wmf.24,Creative Commons Attribution-Share Alike 4.0,https://creativecommons.org/licenses/by-sa/4.0/deed.af,2026-10-23T13:29:23+00:00
23,ig,49265,32747881,664.7,67407,481804,7.15,41282,126,4,21,0,ig,MediaWiki 1.47.0-wmf.24,Creative Commons Attribution-Share Alike 4.0,https://creativecommons.org/licenses/by-sa/4.0/deed.ig,2026-10-23T13:29:41+00:00
56,sw,131758,22063530,167.5,243774,1606785,6.59,92008,216,14,102,1212,sw,MediaWiki 1.47.0-wmf.24,Creative Commons Attribution-Share Alike 4.0,https://creativecommons.org/licenses/by-sa/4.0/deed.sw,2026-10-23T13:30:12+00:00
38,mg,104217,12386749,118.9,263609,1151321,4.37,41156,65,2,28,3,mg,MediaWiki 1.47.0-wmf.24,Creative Commons Attribution-Share Alike 4.0,https://creativecommons.org/licenses/by-sa/4.0/deed.mg,2026-10-23T13:29:54+00:00
15,ff,22583,9572210,423.9,39165,185487,4.74,14648,45,6,15,0,ff,MediaWiki 1.47.0-wmf.24,Creative Commons Attribution-Share Alike 4.0,https://creativecommons.org/licenses/by-sa/4.0/deed.ff,2026-10-23T13:29:35+00:00
66,yo,41387,7135793,172.4,66708,634081,9.51,34966,77,5,35,160,yo,MediaWiki 1.47.0-wmf.24,Creative Commons Attribution-Share Alike 4.0,https://creativecommons.org/licenses/by-sa/4.0/deed.yo,2026-10-23T13:30:23+00:00


## Étape 4: Autres projets Wikimedia par langue

Pour chaque langue retenue :
- les projets publiés de la sitematrix (Wiktionnaire, Wikibooks, Wikiquote, Wikisource, Wikinews, Wikiversité, Wikivoyage), avec leur état (ouvert / fermé) et leur nombre de pages de contenu ;
- les projets **en test dans l'Incubator** (https://incubator.wikimedia.org), pas encore publiés ;
- le nombre de **lexèmes Wikidata** de la langue (données sous CC0), compté par la recherche `haslemma:<code>` de Wikidata.

In [8]:
autres = projets[(projets["code"].isin(editions["code"])) & (projets["projet"] != "wiki")].copy()
contenu = []
for _, p in autres.iterrows():
    d = api(p["url"], f"siteinfo_{p['dbname']}", action="query", meta="siteinfo", siprop="statistics")
    contenu.append(d["query"]["statistics"].get("articles") if d else None)
autres["pages_contenu"] = contenu
autres["resume"] = autres.apply(
    lambda r: f"{r['etat']} ({'?' if pd.isna(r['pages_contenu']) else int(r['pages_contenu'])})", axis=1)
projets_par_langue = autres.pivot_table(index="code", columns="projet", values="resume", aggfunc="first")

# Lexèmes Wikidata par langue  recherche « haslemma:<code> » dans l'espace Lexeme (146) de Wikidata.
# (La requête SPARQL « dct:language » ne renvoie plus de lexèmes depuis le point d'accès principal.)
def compter_lexemes(code):
    d = api("https://www.wikidata.org", f"wikidata_lexemes_{code}", action="query", list="search",
            srsearch=f"haslemma:{code}", srnamespace="146", srlimit="1", srinfo="totalhits")
    return d["query"]["searchinfo"]["totalhits"] if d else None

editions["lexemes_wikidata"] = editions["code"].map(compter_lexemes)

print("Projets hors Wikipédia trouvés :", len(autres))
projets_par_langue.fillna("—")

Projets hors Wikipédia trouvés : 61


projet,wikibooks,wikinews,wikiquote,wikisource,wikiversity,wikivoyage,wiktionary
code,,,,,,,
aa,fermé (0),—,—,—,—,—,fermé (0)
af,ouvert (34),—,ouvert (35),—,—,—,ouvert (30222)
ak,fermé (0),—,—,—,—,—,fermé (0)
am,—,—,fermé (2),—,—,—,ouvert (796)
ar,ouvert (723),fermé (3392),ouvert (3410),ouvert (94672),ouvert (930),—,ouvert (391296)
bm,fermé (0),—,fermé (0),—,—,—,fermé (0)
fr,ouvert (22205),fermé (24204),ouvert (11107),ouvert (719193),ouvert (17627),ouvert (10236),ouvert (7031239)
guw,—,fermé (1045),ouvert (840),—,—,—,ouvert (1748)
ha,—,—,—,—,—,—,ouvert (4176)


### Projets en test dans l'Incubator

L'Incubator héberge les projets en test sous des préfixes : `Wp/<code>/` (Wikipédia), `Wt/` (Wiktionnaire), `Wb/` (Wikibooks), `Wq/` (Wikiquote), `Wn/` (Wikinews) et `Wy/` (Wikivoyage). On compte les pages de chaque préfixe (`list=allpages&apprefix=...`).

Au-delà de `INCUBATOR_MAX_REQUESTS` requêtes, le décompte est plafonné et affiché « ≥ N ». Les tests de Wikisource et de Wikiversité sont hébergés ailleurs (oldwikisource, beta.wikiversity) et ne sont pas couverts.

In [9]:
INCUBATOR = "https://incubator.wikimedia.org"
PREFIXES_INCUBATOR = {"Wp": "wikipedia", "Wt": "wiktionary", "Wb": "wikibooks",
                      "Wq": "wikiquote", "Wn": "wikinews", "Wy": "wikivoyage"}


def compter_incubator(prefixe, code):
    """Renvoie (nombre de pages, plafonné ?) ou (None, False) en cas d'échec."""
    n_pages, cont = 0, {}
    for n in range(INCUBATOR_MAX_REQUESTS):
        d = api(INCUBATOR, f"incubator_{prefixe}_{code}_{n}", action="query", list="allpages",
                apnamespace="0", apprefix=f"{prefixe}/{code}/", aplimit="max", **cont)
        if d is None:
            return None, False
        n_pages += len(d["query"]["allpages"])
        if "continue" not in d:
            return n_pages, False
        cont = d["continue"]
    return n_pages, True


incub = []
for code in editions["code"]:
    for prefixe, projet in PREFIXES_INCUBATOR.items():
        n_pages, plafonne = compter_incubator(prefixe, code)
        if n_pages:
            incub.append({"code": code, "projet": projet,
                          "pages": f"≥ {n_pages}" if plafonne else str(n_pages)})
        elif n_pages is None:
            incub.append({"code": code, "projet": projet, "pages": "échec requête"})
incubator = pd.DataFrame(incub, columns=["code", "projet", "pages"])
incubator_par_langue = incubator.pivot_table(index="code", columns="projet", values="pages", aggfunc="first")
print("Projets en test trouvés dans l'Incubator :", len(incubator))
incubator_par_langue.fillna("—")

Projets en test trouvés dans l'Incubator : 108


projet,wikibooks,wikinews,wikipedia,wikiquote,wikivoyage,wiktionary
code,,,,,,
aa,4,—,169,2,3,26
af,—,19,—,—,73,—
am,7,2,—,13,1,—
ann,—,—,—,—,—,34
ar,—,—,—,—,1308,—
ary,1,—,—,1,—,1857
arz,2,1,—,1,3,≥ 2500
bm,4,—,—,4,—,≥ 2500
bol,—,—,617,—,—,8


## Étape 5  APIs disponibles

Pour chaque édition, on vérifie que trois points d'accès répondent :

| API | Adresse | Test |
|---|---|---|
| **API MediaWiki (Action API)** | `https://<code>.wikipedia.org/w/api.php` | réutilise l'appel `siteinfo` de l'étape 3 |
| **API REST de MediaWiki** | `https://<code>.wikipedia.org/w/rest.php/v1/` | recherche d'une page (`search/page?q=a&limit=1`) |
| **API REST Wikimedia (RESTBase)** | `https://<code>.wikipedia.org/api/rest_v1/` | résumé d'une page au hasard (`page/random/summary`) |

La bibliothèque **Wikipedia-API** s'appuie sur l'API MediaWiki : elle est donc utilisable dès que celle-ci répond. Wikimedia annonce le retrait progressif de RESTBase au profit de l'API REST de MediaWiki.

In [10]:
apis = []
ok_action = set(stats.loc[stats["articles"].notna(), "code"])
for _, e in editions.iterrows():
    base, code = e["url"], e["code"]
    rest = get(f"{base}/w/rest.php/v1/search/page", {"q": "a", "limit": "1"}, label=f"api_rest_{code}")
    restbase = get(f"{base}/api/rest_v1/page/random/summary", label=f"api_restbase_{code}")
    apis.append({
        "code": code,
        "api_action_url": f"{base}/w/api.php", "api_action_ok": code in ok_action,
        "api_rest_url": f"{base}/w/rest.php/v1/", "api_rest_ok": rest is not None,
        "api_restbase_url": f"{base}/api/rest_v1/", "api_restbase_ok": restbase is not None,
    })
apis = pd.DataFrame(apis)
print(apis[["api_action_ok", "api_rest_ok", "api_restbase_ok"]].sum().to_string())
apis

api_action_ok      69
api_rest_ok        69
api_restbase_ok    68


,code,api_action_url,api_action_ok,api_rest_url,api_rest_ok,api_restbase_url,api_restbase_ok
0,aa,https://aa.wikipedia.org/w/api.php,True,https://aa.wikipedia.org/w/rest.php/v1/,True,https://aa.wikipedia.org/api/rest_v1/,True
1,af,https://af.wikipedia.org/w/api.php,True,https://af.wikipedia.org/w/rest.php/v1/,True,https://af.wikipedia.org/api/rest_v1/,True
2,ak,https://ak.wikipedia.org/w/api.php,True,https://ak.wikipedia.org/w/rest.php/v1/,True,https://ak.wikipedia.org/api/rest_v1/,True
3,am,https://am.wikipedia.org/w/api.php,True,https://am.wikipedia.org/w/rest.php/v1/,True,https://am.wikipedia.org/api/rest_v1/,True
4,ann,https://ann.wikipedia.org/w/api.php,True,https://ann.wikipedia.org/w/rest.php/v1/,True,https://ann.wikipedia.org/api/rest_v1/,True
5,ar,https://ar.wikipedia.org/w/api.php,True,https://ar.wikipedia.org/w/rest.php/v1/,True,https://ar.wikipedia.org/api/rest_v1/,True
6,ary,https://ary.wikipedia.org/w/api.php,True,https://ary.wikipedia.org/w/rest.php/v1/,True,https://ary.wikipedia.org/api/rest_v1/,True
7,arz,https://arz.wikipedia.org/w/api.php,True,https://arz.wikipedia.org/w/rest.php/v1/,True,https://arz.wikipedia.org/api/rest_v1/,True
8,bm,https://bm.wikipedia.org/w/api.php,True,https://bm.wikipedia.org/w/rest.php/v1/,True,https://bm.wikipedia.org/api/rest_v1/,True
9,bol,https://bol.wikipedia.org/w/api.php,True,https://bol.wikipedia.org/w/rest.php/v1/,True,https://bol.wikipedia.org/api/rest_v1/,True


## Étape 6: Dumps disponibles

Pour chaque édition, on lit l'index `https://dumps.wikimedia.org/<dbname>/`, puis le fichier `dumpstatus.json` des dates les plus récentes. On retient la plus récente dont le job `articlesdump` (pages-articles) est terminé (`done`).

On relève la date, le nombre de fichiers, la taille totale et l'URL. **Rien n'est téléchargé à cette étape.** Le répertoire `latest/` n'est pas utilisé, car son contenu change chaque mois.

In [11]:
def dernier_dump(dbname, n_dates=4):
    index = get(f"https://dumps.wikimedia.org/{dbname}/", label=f"dumps_index_{dbname}", expect_json=False)
    if index is None:
        return {"dump_statut": "index indisponible"}
    dates = sorted(set(re.findall(r'href="(\d{8})/"', index)), reverse=True)
    for date in dates[:n_dates]:
        st = get(f"https://dumps.wikimedia.org/{dbname}/{date}/dumpstatus.json",
                 label=f"dumpstatus_{dbname}_{date}")
        job = ((st or {}).get("jobs") or {}).get("articlesdump") or {}
        if job.get("status") == "done":
            files = job.get("files") or {}
            return {
                "dump_date": date,
                "dump_nb_fichiers": len(files),
                "dump_taille_mo": round(sum(f.get("size", 0) for f in files.values()) / 1e6, 1),
                "dump_url": "https://dumps.wikimedia.org" + next(iter(files.values()))["url"] if files else None,
                "dump_sha1_1er_fichier": next(iter(files.values())).get("sha1") if files else None,
                "dump_dates_disponibles": len(dates),
                "dump_statut": "ok",
            }
    return {"dump_statut": "aucun dump pages-articles terminé parmi les dates récentes",
            "dump_dates_disponibles": len(dates)}

dumps = pd.DataFrame([{"code": e["code"], **dernier_dump(e["dbname"])} for _, e in editions.iterrows()])
dumps

,code,dump_date,dump_nb_fichiers,dump_taille_mo,dump_url,dump_sha1_1er_fichier,dump_dates_disponibles,dump_statut
0,aa,20261001,1,0.0,https://dumps.wikimedia.org/aawiki/20261001/aawiki-20261001-pages-articles.x...,c1889f0b563970088dcef3478a23f3a2c353c4a0,10,ok
1,af,20261001,1,157.8,https://dumps.wikimedia.org/afwiki/20261001/afwiki-20261001-pages-articles.x...,3cca3bd74247235551c319250c77f8270b71045b,10,ok
2,ak,20261001,1,0.3,https://dumps.wikimedia.org/akwiki/20261001/akwiki-20261001-pages-articles.x...,d2c43fc800d09822622200d59c85ff4b0d893bcf,10,ok
3,am,20261001,1,9.2,https://dumps.wikimedia.org/amwiki/20261001/amwiki-20261001-pages-articles.x...,c50221525e3347ff70787ecdf22c6f30951b786d,10,ok
4,ann,20261001,1,0.7,https://dumps.wikimedia.org/annwiki/20261001/annwiki-20261001-pages-articles...,5a8da1479e640154153b09a08ac15e77b84e1fce,10,ok
5,ar,20261001,10,2025.9,https://dumps.wikimedia.org/arwiki/20261001/arwiki-20261001-pages-articles1....,f99be4f8cf1b8565f35942d813578b5919929931,10,ok
6,ary,20261001,1,18.4,https://dumps.wikimedia.org/arywiki/20261001/arywiki-20261001-pages-articles...,1345b47056ad653ed8f0498dc0840f7b9c6f92e8,10,ok
7,arz,20261001,1,299.3,https://dumps.wikimedia.org/arzwiki/20261001/arzwiki-20261001-pages-articles...,8e13d9b2235163d878fef1cf62e9c462c294a040,10,ok
8,bm,20261001,1,0.9,https://dumps.wikimedia.org/bmwiki/20261001/bmwiki-20261001-pages-articles.x...,f4b46993e61fb895ef278c1b0f336b0472f177e3,10,ok
9,bol,20261001,1,1.7,https://dumps.wikimedia.org/bolwiki/20261001/bolwiki-20261001-pages-articles...,1c782865dc9bb86bb17be9521bc0117b17efdd67,2,ok


## Étape 7  Catégories principales

Source : `list=allcategories&acprop=size|hidden`. Toutes les catégories sont lues (pagination), puis classées par nombre de pages. Les **catégories cachées**, qui servent généralement à la maintenance, sont écartées.

Pour les très grandes éditions, la lecture s'arrête après `CATEGORIES_MAX_REQUESTS` requêtes. Le classement est alors **partiel** et signalé comme tel.

In [12]:
def top_categories(base_url, code):
    cats, n, partiel, cont = [], 0, False, {}
    while True:
        if n >= CATEGORIES_MAX_REQUESTS:
            partiel = True
            break
        d = api(base_url, f"allcategories_{code}_{n}", action="query", list="allcategories",
                aclimit="max", acprop="size|hidden", **cont)
        n += 1
        if d is None:
            break
        cats += d["query"]["allcategories"]
        if "continue" not in d:
            break
        cont = d["continue"]
    visibles = [c for c in cats if not c.get("hidden")]
    top = sorted(visibles, key=lambda c: c.get("pages", 0), reverse=True)[:CATEGORIES_TOP_N]
    return {"code": code, "categories_lues": len(cats), "categories_visibles": len(visibles),
            "categories_classement": "partiel" if partiel else "complet",
            "top_categories": " ; ".join(f"{c['category']} ({c.get('pages', 0)})" for c in top)}

categories = pd.DataFrame([top_categories(e["url"], e["code"]) for _, e in editions.iterrows()])
categories

,code,categories_lues,categories_visibles,categories_classement,top_categories
0,aa,31,31,complet,User aa-0 (22) ; User en (21) ; User en-3 (13) ; User fr (9) ; User en-2 (8)...
1,af,10000,9457,partiel,Flora van Suid-Afrika (10249) ; Amerikaanse manlike akteurs van die 20ste ee...
2,ak,236,235,complet,User en (160) ; Wikipedia bots (77) ; User de (60) ; User fr (53) ; Language...
3,am,1892,1891,complet,ተረትና ምሳሌ (3524) ; ያልተተረጎመ ምሳሌ (2220) ; ፈረንጅ አመቶች (2021) ; ታሪክ ነክ መዋቅሮች (632)...
4,ann,101,101,complet,Pages with script errors (153) ; Ido (86) ; Pages using duplicate arguments ...
5,ar,10000,9654,partiel,آثار العراق (535) ; آثار فلسطين (435) ; 2009 حسب اليوم (375) ; 2008 حسب اليو...
6,ary,10000,9931,partiel,CS1 maint: bot: original URL status unknown (6611) ; الاستشهاد بمصادر ب الإن...
7,arz,10000,9970,partiel,11 دورى كورة قدم سكوتلاند (416) ; All country data templates (415) ; Article...
8,bm,415,413,complet,Pages utilisant des liens magiques ISBN (420) ; Yiri (165) ; Faransi ka dugu...
9,bol,230,230,complet,Pages using template databox (533) ; Templates using TemplateData (329) ; Al...


## Étape 8 Répartition des longueurs d'articles (échantillon via Wikipedia-API)

Le **volume total** de texte vient du compteur exact de l'étape 3 (`mots_articles`). Ce total ne dit pas comment les mots se répartissent : quelques longs articles peuvent cacher une majorité d'articles très courts. L'échantillon sert à mesurer cette répartition, avec la bibliothèque **Wikipedia-API** :
1. `wiki.random(limit=SAMPLE_SIZE)` tire au hasard des articles de l'espace principal (hors redirections) ;
2. `page.text` récupère le texte brut de chaque article ;
3. on compte les mots (découpage sur les espaces) et les caractères.

Résultats par édition : répartition du nombre de mots (min, quartiles, médiane, moyenne, max) et **part d'articles très courts** (moins de `SEUIL_COURT_MOTS` mots). La moyenne de l'échantillon est comparée à la moyenne exacte (`mots_par_article`, étape 3), ce qui indique si l'échantillon est représentatif. Les titres tirés et les textes sont sauvegardés, pour la provenance.

Les requêtes de la bibliothèque sont journalisées grâce à un *hook* httpx, puisque la bibliothèque transmet ses arguments à `httpx.Client`.

In [13]:
def _log_wikipediaapi(response):
    PROVENANCE.append({"n": len(PROVENANCE), "label": "wikipediaapi", "url": str(response.request.url),
                       "horodatage_utc": _now(), "statut_http": response.status_code, "fichier_brut": None})


def echantillon(code):
    wiki = wikipediaapi.Wikipedia(user_agent=USER_AGENT, language=code,
                                  extract_format=wikipediaapi.ExtractFormat.WIKI,
                                  timeout=60.0, event_hooks={"response": [_log_wikipediaapi]})
    try:
        pages = wiki.random(limit=SAMPLE_SIZE)
    except wikipediaapi.WikipediaException as e:
        return [], str(e)
    articles = []
    for titre, page in pages.items():
        try:
            texte = page.text
        except wikipediaapi.WikipediaException as e:
            articles.append({"code": code, "titre": titre, "erreur": str(e)})
            continue
        time.sleep(PAUSE_S)
        n = len(PROVENANCE)
        PROVENANCE.append({"n": n, "label": f"texte_{code}", "url": f"https://{code}.wikipedia.org/wiki/{quote(titre.replace(' ', '_'))}",
                           "horodatage_utc": _now(), "statut_http": 200,
                           "fichier_brut": _save_raw(n, f"texte_{code}", texte, is_json=False)})
        articles.append({"code": code, "titre": titre, "mots": len(texte.split()), "caracteres": len(texte)})
    return articles, None


echantillons, erreurs_echantillon = [], {}
for i, code in enumerate(editions["code"], 1):
    arts, err = echantillon(code)
    echantillons += arts
    if err:
        erreurs_echantillon[code] = err
    print(f"{i}/{len(editions)} {code} : {len(arts)} articles", "" if not err else f"(erreur : {err})")
echantillons = pd.DataFrame(echantillons)

1/69 aa : 1 articles 
2/69 af : 30 articles 
3/69 ak : 1 articles 
4/69 am : 30 articles 
5/69 ann : 30 articles 
6/69 ar : 30 articles 
7/69 ary : 30 articles 
8/69 arz : 30 articles 
9/69 bm : 30 articles 
10/69 bol : 30 articles 
11/69 dag : 30 articles 
12/69 dga : 30 articles 
13/69 din : 30 articles 
14/69 ee : 30 articles 
15/69 fat : 30 articles 
16/69 ff : 30 articles 
17/69 fon : 30 articles 
18/69 fr : 30 articles 
19/69 gpe : 30 articles 
20/69 gur : 30 articles 
21/69 guw : 30 articles 
22/69 ha : 30 articles 
23/69 hz : 1 articles 
24/69 ig : 30 articles 
25/69 igl : 30 articles 
26/69 kab : 30 articles 
27/69 kai : 30 articles 
28/69 kaj : 30 articles 
29/69 kbp : 30 articles 
30/69 kcg : 30 articles 
31/69 kg : 30 articles 
32/69 ki : 30 articles 
33/69 kj : 6 articles 
34/69 knc : 30 articles 
35/69 kr : 1 articles 
36/69 kus : 30 articles 
37/69 lg : 30 articles 
38/69 ln : 30 articles 
39/69 mg : 30 articles 
40/69 mos : 30 articles 
41/69 ng : 21 articles 
42/69 nqo

In [14]:
volume = (echantillons.dropna(subset=["mots"]).groupby("code")["mots"]
          .describe()[["count", "min", "25%", "50%", "mean", "75%", "max"]]
          .rename(columns={"count": "echantillon_n", "min": "mots_min", "25%": "mots_q1", "50%": "mots_mediane",
                           "mean": "mots_moyenne", "75%": "mots_q3", "max": "mots_max"})
          .round(1).reset_index())
courts = (echantillons.dropna(subset=["mots"]).groupby("code")["mots"]
          .apply(lambda s: round(100 * (s < SEUIL_COURT_MOTS).mean(), 1))
          .rename("part_tres_courts_pct").reset_index())
volume = (volume.merge(courts, on="code", how="left")
          .merge(stats[["code", "mots_par_article"]], on="code", how="left"))
volume["ecart_moyenne_pct"] = ((volume["mots_moyenne"] - volume["mots_par_article"])
                               / volume["mots_par_article"] * 100).round(1)
volume.sort_values("part_tres_courts_pct", ascending=False)

,code,echantillon_n,mots_min,mots_q1,mots_mediane,mots_moyenne,mots_q3,mots_max,part_tres_courts_pct,mots_par_article,ecart_moyenne_pct
0,aa,1.0,0.0,0.0,0.0,0.0,0.0,0.0,100.0,NaN,NaN
2,ak,1.0,0.0,0.0,0.0,0.0,0.0,0.0,100.0,NaN,NaN
26,kai,30.0,0.0,11.2,15.0,15.5,17.0,57.0,100.0,62.1,-75.0
34,kr,1.0,0.0,0.0,0.0,0.0,0.0,0.0,100.0,NaN,NaN
32,kj,6.0,0.0,7.0,11.5,18.5,19.0,61.0,100.0,28.5,-35.1
22,hz,1.0,0.0,0.0,0.0,0.0,0.0,0.0,100.0,NaN,NaN
61,tum,30.0,0.0,15.0,20.0,92.6,24.0,1700.0,93.3,155.8,-40.6
43,nso,30.0,1.0,18.0,21.0,45.7,33.5,334.0,90.0,81.2,-43.7
3,am,30.0,3.0,11.2,22.0,50.9,51.5,392.0,86.7,134.6,-62.2
31,ki,30.0,6.0,15.0,42.0,47.7,70.2,129.0,86.7,83.9,-43.1


## Étape 9 Langues africaines en préparation dans l'Incubator

Certaines langues africaines n'ont pas encore de Wikipédia publiée, mais une **Wikipédia en test** dans l'Incubator. Elles sont présentées **à part**, sans être mélangées aux éditions publiées des étapes 2 à 8.

**Méthode**
1. Liste de tous les projets en test : sous-catégories de `Category:Incubator:All test wikis`. Chaque test Wikipédia a une catégorie `Category:Wp/<code>`.
2. Identification des langues africaines : on cherche dans Wikidata les langues « indigènes à » (P2341), ou associées par « pays » (P17), un lieu situé en Afrique. On relève leurs codes ISO 639-3 (P220), ISO 639-1 (P218) et Wikimedia (P424), puis on les compare aux codes des tests.
3. Revue manuelle : exclusion des faux positifs, et signalement des langues des signes et des langues anciennes ou liturgiques.
4. Taille de chaque test : nombre de pages de la catégorie `Category:Wp/<code>` (`prop=categoryinfo`).
5. Les tests de langues qui ont **déjà** une édition publiée (ouverte ou fermée) sont signalés à part.

In [15]:
# 1. Tous les projets en test de l'Incubator
sous_cats, cont, n = [], {}, 0
while True:
    d = api(INCUBATOR, f"incubator_tous_tests_{n}", action="query", list="categorymembers",
            cmtitle="Category:Incubator:All test wikis", cmtype="subcat", cmlimit="max", **cont)
    n += 1
    if d is None:
        break
    sous_cats += [m["title"] for m in d["query"]["categorymembers"]]
    if "continue" not in d:
        break
    cont = d["continue"]
tests = pd.DataFrame([{"projet": m.group(1), "code": m.group(2)} for t in sous_cats
                      if (m := re.match(r"^Category:(W[a-z])/([a-z-]+)$", t))])
tests_wp = sorted(tests.loc[tests["projet"] == "Wp", "code"].unique())
print("Projets en test :", len(tests), "| dont Wikipédia :", len(tests_wp))
print(tests["projet"].value_counts().to_string())

Projets en test : 1482 | dont Wikipédia : 708
projet
Wp    708
Wt    311
Wy    129
Wb    117
Wn    116
Wq    101


In [16]:
# 2. Codes des langues africaines dans Wikidata
q_codes_afrique = """
SELECT DISTINCT ?lang ?code ?nom_fr ?nom_en WHERE {
  { ?lang wdt:P2341 ?place } UNION { ?lang wdt:P17 ?place }
  { ?place wdt:P30 wd:Q15 } UNION { ?place wdt:P17 ?pays . ?pays wdt:P30 wd:Q15 }
  { ?lang wdt:P220 ?code } UNION { ?lang wdt:P218 ?code } UNION { ?lang wdt:P424 ?code }
  OPTIONAL { ?lang rdfs:label ?nom_fr FILTER(LANG(?nom_fr) = "fr") }
  OPTIONAL { ?lang rdfs:label ?nom_en FILTER(LANG(?nom_en) = "en") }
}"""
codes_afrique = (pd.DataFrame(sparql_rows(sparql(q_codes_afrique, "wikidata_codes_langues_afrique")))
                 .assign(lang=lambda df: df["lang"].map(qid))
                 .groupby("code").agg(qid_langue=("lang", "first"), nom_francais=("nom_fr", "first"),
                                      nom_anglais=("nom_en", "first")))
print("Codes de langues africaines dans Wikidata :", len(codes_afrique))

# 3. Revue manuelle (23/10/2026)
EXCLUSIONS_INCUBATOR = {
    "lzz": "faux positif : laze, langue de Turquie et de Géorgie",
    "ayh": "faux positif : arabe hadrami, langue du Yémen",
}
REMARQUES_INCUBATOR = {
    "tse": "langue des signes : pas de texte écrit propre",
    "esl": "langue des signes : pas de texte écrit propre",
    "cop": "langue liturgique (copte), plus parlée au quotidien",
}

publiees = dict(zip(wikipedias["code"], wikipedias["etat"]))
prep = []
for code in tests_wp:
    if code not in codes_afrique.index or code in EXCLUSIONS_INCUBATOR:
        continue
    info = codes_afrique.loc[code]
    prep.append({
        "code": code, "nom_francais": info["nom_francais"], "nom_anglais": info["nom_anglais"],
        "qid_langue": info["qid_langue"],
        "edition_publiee": publiees.get(code, "aucune"),
        "remarque": REMARQUES_INCUBATOR.get(code, ""),
        "url_test": f"{INCUBATOR}/wiki/Wp/{code}",
    })
en_preparation = pd.DataFrame(prep, columns=["code", "nom_francais", "nom_anglais", "qid_langue",
                                             "edition_publiee", "remarque", "url_test"])
print("Tests Wikipédia en langues africaines :", len(en_preparation))
print("Exclus (faux positifs) :", sorted(set(tests_wp) & set(EXCLUSIONS_INCUBATOR)))

Codes de langues africaines dans Wikidata : 2326
Tests Wikipédia en langues africaines : 121
Exclus (faux positifs) : ['ayh', 'lzz']


In [17]:
# 4. Taille de chaque test (pages de la catégorie Wp/<code>)
titres = [f"Category:Wp/{c}" for c in en_preparation["code"]]
taille = {}
for i in range(0, len(titres), 50):
    d = api(INCUBATOR, f"incubator_categoryinfo_{i // 50}", action="query", prop="categoryinfo",
            titles="|".join(titres[i:i + 50]))
    for p in (d or {}).get("query", {}).get("pages", []):
        taille[p["title"].split("/", 1)[1]] = (p.get("categoryinfo") or {}).get("pages")
en_preparation["pages_test"] = en_preparation["code"].map(taille)

# 5. Séparation : vraies langues en préparation / tests de langues déjà publiées
deja_publiees = en_preparation[en_preparation["edition_publiee"] != "aucune"]
en_preparation = (en_preparation[en_preparation["edition_publiee"] == "aucune"]
                  .sort_values("pages_test", ascending=False).reset_index(drop=True))
print("Langues africaines en préparation (sans Wikipédia publiée) :", len(en_preparation))
print("Tests restants de langues déjà publiées :", deja_publiees["code"].tolist())
en_preparation[["code", "nom_francais", "nom_anglais", "pages_test", "remarque", "url_test"]]

Langues africaines en préparation (sans Wikipédia publiée) : 114
Tests restants de langues déjà publiées : ['aa', 'bol', 'fat', 'fon', 'hz', 'kj', 'ng']


,code,nom_francais,nom_anglais,pages_test,remarque,url_test
0,lua,luba-kasaï,Luba-Kasai,2881,,https://incubator.wikimedia.org/wiki/Wp/lua
1,kri,krio,Krio,1695,,https://incubator.wikimedia.org/wiki/Wp/kri
2,cop,copte,Coptic,1517,"langue liturgique (copte), plus parlée au quotidien",https://incubator.wikimedia.org/wiki/Wp/cop
3,wlx,wali,Wali,1487,,https://incubator.wikimedia.org/wiki/Wp/wlx
4,arq,arabe algérien,Algerian Arabic,1240,,https://incubator.wikimedia.org/wiki/Wp/arq
5,gaa,ga,Ga,1223,,https://incubator.wikimedia.org/wiki/Wp/gaa
6,vmw,makua,Makhuwa,1006,,https://incubator.wikimedia.org/wiki/Wp/vmw
7,bci,baoulé,Baoulé,954,,https://incubator.wikimedia.org/wiki/Wp/bci
8,tzm,tamazight du Maroc central,Central Atlas Tamazight,679,,https://incubator.wikimedia.org/wiki/Wp/tzm
9,wal,wolaita,Wolayttattuwa,677,,https://incubator.wikimedia.org/wiki/Wp/wal


## Étape 10 Synthèse, contrôles automatiques et sauvegarde

Fusion des étapes 2 à 8 en un tableau par édition publiée (les langues en préparation de l'étape 9 sont sauvegardées à part), puis des contrôles automatiques :
- statistiques et nombre de mots obtenus pour chaque édition ;
- API MediaWiki disponible pour chaque édition ;
- provenance renseignée pour chaque requête ;
- échantillon obtenu pour chaque édition ouverte ;
- aucune nouvelle détection Wikidata en attente de revue.

**Fichiers produits** dans `data/processed/inventaire/<RUN_ID>/` :
- `inventaire_synthese.json` : le tableau final ;
- `echantillon_articles.json` : les titres tirés et leurs longueurs ;
- `langues_en_preparation_incubator.json` : les langues africaines qui n'ont qu'une Wikipédia en test (étape 9) ;
- `provenance.jsonl` : le journal de toutes les requêtes ;
- `run_metadata.json` : la date, les versions et les paramètres du run.

In [18]:
synthese = (editions[["code", "nom_francais", "nom_local", "nom_anglais", "dbname", "url", "etat", "qid_langue", "iso639_3",
                      "critere", "decision", "commentaire", "langue_initiale", "lexemes_wikidata"]]
            .merge(stats, on="code", how="left")
            .merge(projets_par_langue.add_prefix("projet_").reset_index(), on="code", how="left")
            .merge(incubator_par_langue.add_prefix("incubator_").reset_index(), on="code", how="left")
            .merge(apis, on="code", how="left")
            .merge(dumps, on="code", how="left")
            .merge(categories, on="code", how="left")
            .merge(volume.drop(columns=["mots_par_article"]), on="code", how="left")
            .sort_values("mots_articles", ascending=False).reset_index(drop=True))

# Contrôles automatiques 
controles = {
    "statistiques pour chaque édition": bool(synthese["articles"].notna().all()),
    "nombre de mots pour chaque édition": bool(synthese["mots_articles"].notna().all()),
    "API MediaWiki disponible pour chaque édition": bool(synthese["api_action_ok"].fillna(False).all()),
    "provenance pour chaque requête": all(p.get("url") and p.get("horodatage_utc") for p in PROVENANCE),
    "aucune requête en échec": not any("erreur" in p for p in PROVENANCE),
    "échantillon pour chaque édition ouverte": set(synthese.loc[synthese["etat"] == "ouvert", "code"])
                                               <= set(echantillons.get("code", pd.Series(dtype=str))),
    "dump trouvé pour chaque édition": bool((synthese["dump_statut"] == "ok").all()),
    "aucune nouvelle détection à revoir": not synthese["commentaire"].str.contains("à revoir").any(),
}
for nom, ok in controles.items():
    print("OK   " if ok else "ÉCHEC", nom)
echecs = [p for p in PROVENANCE if "erreur" in p]
if echecs:
    print(f"\n{len(echecs)} requête(s) en échec :")
    for p in echecs[:20]:
        print(" -", p["label"], p["erreur"])

# Sauvegarde 
synthese.to_json(OUT_DIR / "inventaire_synthese.json", orient="records", force_ascii=False, indent=2)
echantillons.to_json(OUT_DIR / "echantillon_articles.json", orient="records", force_ascii=False, indent=2)
en_preparation.to_json(OUT_DIR / "langues_en_preparation_incubator.json", orient="records", force_ascii=False, indent=2)
with open(OUT_DIR / "provenance.jsonl", "w", encoding="utf-8") as f:
    for p in PROVENANCE:
        f.write(json.dumps(p, ensure_ascii=False) + "\n")
metadata = {
    "run_id": RUN_ID, "fin_utc": _now(), "python": sys.version.split()[0], "plateforme": platform.platform(),
    "pandas": pd.__version__, "requests": requests.__version__,
    "wikipedia_api": ".".join(map(str, wikipediaapi.__version__)) if isinstance(wikipediaapi.__version__, tuple) else str(wikipediaapi.__version__),
    "parametres": {"SAMPLE_SIZE": SAMPLE_SIZE, "SEUIL_COURT_MOTS": SEUIL_COURT_MOTS,
                   "CATEGORIES_TOP_N": CATEGORIES_TOP_N, "CATEGORIES_MAX_REQUESTS": CATEGORIES_MAX_REQUESTS,
                   "INCUBATOR_MAX_REQUESTS": INCUBATOR_MAX_REQUESTS, "PAUSE_S": PAUSE_S},
    "nb_requetes": len(PROVENANCE), "nb_echecs": len(echecs), "controles": controles,
    "editions": {"inventoriees": len(synthese), "decisions": synthese["decision"].value_counts().to_dict()},
    "incubator": {"tests_wikipedia": len(tests_wp), "langues_africaines_en_preparation": len(en_preparation)},
}
with open(OUT_DIR / "run_metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)
print("\nRésultats sauvegardés dans", OUT_DIR)
synthese[["code", "nom_francais", "etat", "decision", "articles", "mots_articles", "mots_par_article",
          "mots_mediane", "part_tres_courts_pct", "utilisateurs_actifs", "bots",
          "api_rest_ok", "dump_date", "dump_taille_mo"]]

OK    statistiques pour chaque édition
OK    nombre de mots pour chaque édition
OK    API MediaWiki disponible pour chaque édition
OK    provenance pour chaque requête
ÉCHEC aucune requête en échec
OK    échantillon pour chaque édition ouverte
OK    dump trouvé pour chaque édition
OK    aucune nouvelle détection à revoir

1 requête(s) en échec :
 - api_restbase_om HTTP 400

Résultats sauvegardés dans C:\Users\akota\desktop\projet_stage_nlp\data\processed\inventaire\20261023T132917Z


,code,nom_francais,etat,decision,articles,mots_articles,mots_par_article,mots_mediane,part_tres_courts_pct,utilisateurs_actifs,bots,api_rest_ok,dump_date,dump_taille_mo
0,fr,français,ouvert,référence,2786157,1932510322,693.6,148.5,33.3,37142,63,True,20261001,7023.7
1,ar,arabe standard moderne,ouvert,à décider,1338075,506530597,378.6,94.5,53.3,5575,19,True,20261001,2025.9
2,arz,arabe égyptien,ouvert,inclus,1633890,269346473,164.8,102.5,43.3,290,34,True,20261001,299.3
3,ha,haoussa,ouvert,inclus,121191,77392768,638.6,251.0,30.0,286,15,True,20261001,186.2
4,af,afrikaans,ouvert,inclus,138988,47372484,340.8,121.0,40.0,273,65,True,20261001,157.8
5,ig,igbo,ouvert,inclus,49265,32747881,664.7,233.0,13.3,126,21,True,20261001,87.4
6,sw,swahili,ouvert,inclus,131758,22063530,167.5,59.0,66.7,216,102,True,20261001,71.7
7,mg,malgache,ouvert,inclus,104217,12386749,118.9,39.0,56.7,65,28,True,20261001,38.1
8,ff,peul,ouvert,inclus,22583,9572210,423.9,153.0,33.3,45,15,True,20261001,23.2
9,yo,yoruba,ouvert,inclus,41387,7135793,172.4,17.5,73.3,77,35,True,20261001,29.5


## Limites connues de cet inventaire

- **Classement « langue africaine »** : il repose sur Wikidata, puis sur une revue manuelle (étape 2). L'arabe standard .
- **Incubator** : les préfixes Wp, Wt, Wb, Wq, Wn et Wy sont vérifiés pour chaque code de langue. Les tests de Wikisource et de Wikiversité, hébergés ailleurs, ne le sont pas. Un test publié sous un autre code (par exemple un code ISO 639-3) peut échapper au décompte.
- **Langues en préparation (étape 9)** : l'identification repose sur la correspondance entre le code du test et les codes Wikidata (ISO 639-1, ISO 639-3, Wikimedia). Un test publié sous un code absent de Wikidata, ou une langue mal renseignée dans Wikidata, peut échapper au décompte. La taille d'un test est le nombre de pages de sa catégorie, pas un nombre d'articles au sens strict.
- **Volume de texte** : le total de mots vient du compteur de CirrusSearch, avec sa propre méthode de découpage. Il est à recouper avec les dumps lors des tâches suivantes.
- **Échantillon** : `SAMPLE_SIZE` articles par édition, uniquement pour la répartition des longueurs. Le seuil « très court » (`SEUIL_COURT_MOTS`) est un paramètre de travail, à ajuster.
- **Comptage des mots dans l'échantillon** : découpage sur les espaces, sans tokenisation propre à chaque langue. Le texte des modèles (infobox, tableaux) n'est pas inclus par `page.text`.
- **Catégories** : classement partiel pour les très grandes éditions (voir la colonne `categories_classement`).
- **Identification de la langue** : la part de texte réellement écrite dans la langue cible n'est pas mesurée ici.
- **Instantané** : les statistiques sont relevées en direct et évoluent. Seul le `RUN_ID` permet de dater les chiffres.